# Residual experiments toward Public AUC 0.97074

## tl;dr

The reproducible public score improved from **0.97061 to 0.97063** by adding TabNet and a second RealMLP as low-weight rank signals. Tree residual correction and additional weak MLP/GBM members did not add meaningful OOF value.

## Context & Methods

The anchor is the 54-model honest OOF stack. Every incremental candidate was evaluated with 5-fold StratifiedKFold, shuffle enabled, seed 42. Blend weights were selected on the other four folds and applied to the held-out fold.

### Key Assumptions

Public OOF files preserve original train row order or expose IDs that can be aligned exactly. Public leaderboard scores are rounded to five decimals.

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'analysis' else Path.cwd()
def load_json(path):
    return json.loads((ROOT / path).read_text())

## Data

Load the saved metrics produced by local diagnostics and Kaggle GPU notebooks.

In [ ]:
metrics = {
    'stack54': load_json('kaggle/oof_stack_v4_output_retry/metrics.json'),
    'xgb_residual': load_json('kaggle/xgb_residual_v8_output/metrics.json'),
    'tabnet': load_json('kaggle/tabnet_blend_v9_output/metrics.json'),
    'realmlp': load_json('kaggle/realmlp_blend_v10_output/metrics.json'),
    'extra_mlp_gbm': load_json('kaggle/danush_blend_v11_output/metrics.json'),
}
sorted(metrics)

## Results

The progression table separates honest OOF evidence from rounded Public LB results.

In [ ]:
progression = pd.DataFrame([
    {'experiment': 'HistGB baseline', 'oof_auc': 0.9590605, 'public_auc': 0.96018},
    {'experiment': 'LGBM + HistGB', 'oof_auc': 0.9643539, 'public_auc': 0.96588},
    {'experiment': '54-model stack', 'oof_auc': metrics['stack54']['blend_oof_auc'], 'public_auc': 0.97061},
    {'experiment': '+ 9.5% TabNet rank', 'oof_auc': metrics['tabnet']['chosen_oof_auc'], 'public_auc': 0.97063},
    {'experiment': '+ 12% RealMLP rank', 'oof_auc': metrics['realmlp']['chosen_oof_auc'], 'public_auc': 0.97063},
])
progression

In [ ]:
diagnostics = pd.DataFrame([
    {'candidate': 'GPU XGB base-margin correction', 'delta_oof': metrics['xgb_residual']['blend_delta'], 'decision': 'reject'},
    {'candidate': 'TabNet rank signal', 'delta_oof': metrics['tabnet']['chosen_delta'], 'decision': 'keep'},
    {'candidate': 'Second RealMLP rank signal', 'delta_oof': metrics['realmlp']['chosen_delta'], 'decision': 'keep'},
    {'candidate': 'Extra MLP rank signal', 'delta_oof': metrics['extra_mlp_gbm']['rank_results']['mlp']['delta'], 'decision': 'reject'},
    {'candidate': 'Extra GBM rank signal', 'delta_oof': metrics['extra_mlp_gbm']['rank_results']['gbm']['delta'], 'decision': 'reject'},
]).sort_values('delta_oof', ascending=False)
diagnostics

## Takeaways

- TabNet and the second RealMLP contributed because their rank correlations with the anchor were below the saturated tree correction signals.
- The current verified score is **0.97063**; the requested 0.97074 remains **0.00011** away.
- Further progress needs a new neural representation with aligned OOF, not more tree tuning or leaderboard-only blending.